# Fine-tune Qwen2-VL-2B trên Google Colab – Nhóm 7

Notebook này train lại QLoRA như vòng 2 trên GPU T4 miễn phí của Colab, đánh giá trước/sau và lưu mọi kết quả lên Google Drive.

**Trước khi chạy:**
1. Vào **Thời gian chạy > Thay đổi loại thời gian chạy** (Runtime > Change runtime type), chọn **T4 GPU**.
2. Đặt file `du_lieu_xu_ly_v2.zip` vào thư mục `nhom7` trong **Drive của tôi**. Nếu Phương có chia sẻ `du_doan_goc.jsonl` thì để chung thư mục đó.
3. Chạy lần lượt từng ô từ trên xuống, đợi ô trước xong mới chạy ô sau.

Mỗi khi phiên Colab bị ngắt hoặc mở lại, phải chạy lại các ô từ đầu đến hết Bước 3. Kết quả đã lưu trên Drive vẫn giữ nguyên.

## Cấu hình

Sửa `TEN_CHAY` cho mỗi lần train, theo dạng `v3-<tên>-<điểm khác>`. Mỗi lần chỉ đổi một tham số so với vòng 2 và ghi vào `THAM_SO`; để trống là giữ cấu hình vòng 2.

In [ ]:
TEN_CHAY = "v3-ten-ban"   # vi du: v3-hai-r32
THAM_SO = ""              # vi du: "--lora_r 32 --lora_alpha 32"
DRIVE = "/content/drive/MyDrive/nhom7"
OUT = f"{DRIVE}/ket_qua/{TEN_CHAY}"
print("Ket qua se luu o:", OUT)

## Bước 1 – Kiểm tra GPU, kết nối Drive

Kết quả đúng: dòng `Tesla T4, 15360 MiB`, rồi cho phép Colab truy cập Drive.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

## Bước 2 – Lấy code và dữ liệu

Kết quả đúng: thấy `llm_train.jsonl`, `llm_val.jsonl`, `vlm_train.jsonl`, `vlm_val.jsonl` và hai thư mục `ingest`, `images`.

In [ ]:
%cd /content
!git clone -q https://github.com/phuongdtrai/Hoc-may-nang-cao.git nhom7 || git -C nhom7 pull -q
%cd /content/nhom7
!unzip -q -o "{DRIVE}/du_lieu_xu_ly_v2.zip" -d .
!ls data/processed

## Bước 3 – Cài môi trường (3–5 phút)

Cả nhóm dùng chung phiên bản thư viện trong `requirements-colab.txt` của repo, giống môi trường đã train vòng 2, chỉ trừ torch. Ô này giữ nguyên torch có sẵn của Colab rồi cài theo file đó. Dòng đỏ "pip's dependency resolver…" liệt kê gói khác của Colab, bỏ qua được. Kết quả đúng: dòng cuối có `5.17.0 1.15.0 2026.10.5`.

In [ ]:
%cd /content/nhom7
import importlib.metadata as md
with open('/content/giu_torch.txt', 'w') as f:
    for goi in ('torch', 'torchvision', 'torchaudio'):
        try:
            f.write(f'{goi}=={md.version(goi)}\n')
        except md.PackageNotFoundError:
            pass
print(open('/content/giu_torch.txt').read())
!pip install -q -c /content/giu_torch.txt -r requirements-colab.txt
!python -c "import torch, transformers, trl, unsloth; print(torch.__version__, transformers.__version__, trl.__version__, unsloth.__version__)"

## Bước 4 – Kiểm tra và chạy thử 10 bước

- `check_env.py` không được có dòng `[FAIL]`.
- `--dry_run` phải kết thúc bằng "Du lieu va prompt on".
- Lần chạy thử in ra "XONG sau … phút (train … phút, 10 buoc)"; số phút train nhân 24 là ước lượng thời gian train đầy đủ.
- Loss phải là số bình thường, không phải `nan`.

In [ ]:
%cd /content/nhom7
!python finetune/check_env.py
!python finetune/train_qwen2vl.py --dry_run
!python finetune/train_qwen2vl.py --max_steps 10 --output_dir /content/thu_10_buoc

## Bước 5 – Train đầy đủ, lưu thẳng lên Drive

Để nguyên tab Colab trong lúc train, không cho máy ngủ. Không dùng `--save_merged`, vì bản gộp nặng khoảng 4,5 GB.

In [ ]:
%cd /content/nhom7
!mkdir -p "{OUT}"
!python -u finetune/train_qwen2vl.py --output_dir "{OUT}/lora" {THAM_SO} 2>&1 | tee -a "{OUT}/train.log"

### Chỉ chạy ô dưới khi bị ngắt giữa lúc train

Trước đó chạy lại ô Cấu hình và Bước 1–3, giữ nguyên `TEN_CHAY` và `THAM_SO`. Script train tiếp từ checkpoint cuối epoch gần nhất.

In [ ]:
%cd /content/nhom7
!python -u finetune/train_qwen2vl.py --output_dir "{OUT}/lora" {THAM_SO} --resume 2>&1 | tee -a "{OUT}/train.log"

## Bước 6 – Đánh giá trước/sau

Ô đầu chạy thử 10 câu. Nếu ổn thì chạy ô thứ hai để đánh giá đủ; các câu đã trả lời ở ô đầu được dùng lại. Nếu có `du_doan_goc.jsonl` trên Drive, script bỏ qua phần mô hình gốc, đỡ gần một nửa thời gian.

In [ ]:
%cd /content/nhom7
import os, shutil
os.makedirs(f"{OUT}/danh_gia", exist_ok=True)
goc = f"{DRIVE}/du_doan_goc.jsonl"
if os.path.exists(goc) and not os.path.exists(f"{OUT}/danh_gia/du_doan_goc.jsonl"):
    shutil.copy(goc, f"{OUT}/danh_gia/")
    print("Dung lai cau tra loi cua mo hinh goc tu vong 2")
!python -u finetune/eval_truoc_sau.py --adapter "{OUT}/lora" --out_dir "{OUT}/danh_gia" --limit 10

In [ ]:
%cd /content/nhom7
!python -u finetune/eval_truoc_sau.py --adapter "{OUT}/lora" --out_dir "{OUT}/danh_gia"

## Bước 7 – Nén adapter để gửi nhóm

Bỏ các thư mục `checkpoint-*`, vì chúng chỉ dùng để train tiếp.

In [ ]:
!cd "{OUT}" && zip -qr "lora_{TEN_CHAY}.zip" lora -x "lora/checkpoint-*" && ls -lh "{OUT}"

## Gửi kết quả

Chia sẻ thư mục `nhom7/ket_qua/<TEN_CHAY>/` trên Drive cho Phương, quyền Người xem là đủ. Thư mục đó có:

- adapter đã nén `lora_<TEN_CHAY>.zip`;
- biểu đồ loss và thông tin train: `lora/loss.png`, `lora/thong_tin_train.json`;
- nhật ký `train.log`;
- kết quả đánh giá: `danh_gia/bao_cao_danh_gia.md`, `danh_gia/so_sanh_truoc_sau.csv`.

`thong_tin_train.json` đã ghi mục `moi_truong`: phiên bản thư viện, GPU, fp16 hay bf16, commit code và mã băm dữ liệu, nên khi so kết quả giữa các máy biết ngay có cùng điều kiện hay không.

Nhắn vào nhóm: `TEN_CHAY`, `THAM_SO`, val loss trước → sau, và F1 của ba nhóm câu trong `bao_cao_danh_gia.md`. Không đẩy adapter hay kết quả lên GitHub.